# Theme 2: Bug-Fix Quality Over Time

**RQ2:** How does bug-fix *acceptance rate* change over time?  
**RQ3:** How does *time to merge* change over time?  
**RQ4:** How does *patch size* change over time?  
**RQ5:** How does *revision burden* change over time?

Dataset: `mabujadallah/GitHub-Agentic-PR-Dataset`  
Coverage: Dec 2024 – Feb 2026 (15 months)

In [1]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: C:\Users\Mahmoudabujadallah\final_replication\GitHub-Agentic-PR-Dataset\.venv\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, chi_square, mann_whitney, sig_label,
    set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME1_DIR, THEME2_DIR, THEME3_DIR,
)
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [3]:
# ── Load data ────────────────────────────────────────────────────────
df         = load_fix_prs()
agents_df  = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df   = df[~df['is_agent']].copy()

# Commit data needed for RQ5
commits   = load_commits()
details   = load_commit_details()
rev_stats = build_revision_stats(df, commits, details)
print('All data loaded.')

Loading fix PRs from HuggingFace ...


  Fix PRs loaded: 404,700  |  Agent: 116,011  |  Human: 288,689


Loading commits from HuggingFace ...


  Commits loaded: 1,156,238
Loading commit details from HuggingFace ...


  Commit details loaded: 7,451,150


All data loaded.


## RQ2: How does bug-fix acceptance rate change over time?

In [4]:
# Monthly merge rate per agent + human baseline
months = sorted(df['month'].unique())
rows   = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = agents_df[(agents_df['month'] == m) & (agents_df['agent'] == agent)]
        row[agent] = round(sub['is_merged'].mean() * 100, 1) if len(sub) >= 5 else None
    h = human_df[human_df['month'] == m]
    row['Human'] = round(h['is_merged'].mean() * 100, 1) if len(h) >= 5 else None
    rows.append(row)
monthly_rate = pd.DataFrame(rows).set_index('month')
print('Monthly merge rate (%):')
print(monthly_rate.to_string())

Monthly merge rate (%):
         Copilot  Cursor  Claude_Code  Devin  Human
month                                              
2024-12     91.3    94.5         83.4   58.0   87.9
2025-01     91.7    96.2         83.5   59.2   87.3
2025-02     95.8    93.6         83.4   59.6   88.0
2025-03     95.3    94.2         78.4   46.5   87.0
2025-04     93.1    94.1         84.5   70.6   88.0
2025-05     73.5    88.8         84.6   75.8   87.0
2025-06     78.3    88.7         87.6   70.5   85.8
2025-07     78.3    89.3         88.1   69.5   84.4
2025-08     79.7    86.9         88.8   77.7   83.9
2025-09     73.5    88.4         82.7   65.8   84.0
2025-10     76.7    89.0         86.5   65.6   83.5
2025-11     77.7    87.3         85.3   52.7   83.9
2025-12     76.3    89.6         89.7   74.3   83.7
2026-01     75.5    88.7         89.7   77.1   82.6
2026-02     75.1    87.5         89.2   69.0   81.7


In [5]:
# Figure: monthly acceptance rate per agent vs human
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_rate.index, monthly_rate[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.plot(monthly_rate.index, monthly_rate['Human'], 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Merge Rate (%)')
ax.set_ylim(0, 105)
ax.set_title('RQ2: Monthly Bug-Fix Acceptance Rate per Agent vs Human')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq2_monthly_merge_rate', THEME2_DIR)

  -> Saved: results\theme2_figures\rq2_monthly_merge_rate.png


WindowsPath('results/theme2_figures/rq2_monthly_merge_rate.png')

## RQ3: How does time to merge change over time?

In [6]:
# Monthly median time-to-merge per agent + human
merged_agents = agents_df[agents_df['is_merged']]
merged_human  = human_df[human_df['is_merged']]
rows_ttm = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = merged_agents[(merged_agents['month'] == m) & (merged_agents['agent'] == agent)]
        row[agent] = round(sub['hours_to_merge'].median(), 2) if len(sub) >= 5 else None
    h = merged_human[merged_human['month'] == m]
    row['Human'] = round(h['hours_to_merge'].median(), 2) if len(h) >= 5 else None
    rows_ttm.append(row)
monthly_ttm = pd.DataFrame(rows_ttm).set_index('month')
print('Monthly median time to merge (hours):')
print(monthly_ttm.to_string())

Monthly median time to merge (hours):
         Copilot  Cursor  Claude_Code  Devin  Human
month                                              
2024-12     2.05    0.56         2.32   0.46   5.27
2025-01     1.98    0.52         2.92   0.31   5.28
2025-02     0.75    0.75         1.03   0.07   4.45
2025-03     4.82    0.52         1.35   0.23   5.56
2025-04     1.81    0.73         1.26   0.20   4.98
2025-05     0.51    0.41         0.80   0.22   5.41
2025-06     0.59    0.15         0.26   0.21   5.10
2025-07     0.52    0.04         0.27   0.23   5.49
2025-08     0.85    0.24         0.64   0.40   6.09
2025-09     1.37    0.53         1.10   1.98   5.70
2025-10     2.32    0.59         1.09   3.10   6.59
2025-11     1.95    0.46         1.20   2.67   6.23
2025-12     1.82    0.54         0.90   1.33   5.34
2026-01     2.51    0.54         0.96   2.28   5.42
2026-02     2.17    0.58         0.69   3.88   5.08


In [7]:
# Figure: monthly time to merge
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_ttm.index, monthly_ttm[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.plot(monthly_ttm.index, monthly_ttm['Human'], 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Time to Merge (hours)')
ax.set_title('RQ3: Monthly Median Time to Merge per Agent vs Human')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq3_monthly_time_to_merge', THEME2_DIR)

  -> Saved: results\theme2_figures\rq3_monthly_time_to_merge.png


WindowsPath('results/theme2_figures/rq3_monthly_time_to_merge.png')

## RQ4: How does bug-fix patch size change over time?

In [8]:
# Aggregate total lines added/deleted per PR
pr_size = (
    details.groupby('pr_id')
    .agg(lines_added=('additions', 'sum'), lines_deleted=('deletions', 'sum'))
    .reset_index()
    .rename(columns={'pr_id': 'id'})
)
df_size       = df.merge(pr_size, on='id', how='left')
agents_size   = df_size[df_size['is_agent'] & df_size['agent'].isin(AGENTS)]
human_size    = df_size[~df_size['is_agent']]

rows_size = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = agents_size[(agents_size['month'] == m) & (agents_size['agent'] == agent)]
        row[agent] = round(sub['lines_added'].median(), 1) if len(sub) >= 5 else None
    h = human_size[human_size['month'] == m]
    row['Human'] = round(h['lines_added'].median(), 1) if len(h) >= 5 else None
    rows_size.append(row)
monthly_size = pd.DataFrame(rows_size).set_index('month')
print('Monthly median lines added:')
print(monthly_size.to_string())

Monthly median lines added:
         Copilot  Cursor  Claude_Code  Devin  Human
month                                              
2024-12     13.5    20.0         11.5   74.0   16.0
2025-01     13.0    14.0         15.0   70.0   16.0
2025-02     13.0    16.0          9.0   30.0   17.0
2025-03     11.0    19.0         10.0   29.0   18.0
2025-04     38.0    20.0         16.0   24.0   19.0
2025-05     38.0    20.0         27.0   37.0   17.0
2025-06     51.0    22.0         28.0   38.5   19.0
2025-07     68.0    62.0         50.0   31.5   20.0
2025-08     90.0    40.0         37.0   34.0   21.0
2025-09     74.0    33.0         30.0   55.0   22.0
2025-10     73.0    35.0         28.0   42.0   22.0
2025-11     60.5    34.0         25.0   81.0   24.0
2025-12     52.0    31.0         27.0   46.0   24.0
2026-01     51.0    35.5         28.0   37.0   25.0
2026-02     34.0    42.0         30.0   39.0   29.0


In [9]:
# Figure: monthly patch size (lines added)
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_size.index, monthly_size[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.plot(monthly_size.index, monthly_size['Human'], 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Lines Added')
ax.set_title('RQ4: Monthly Median Patch Size (Lines Added) per Agent vs Human')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq4_monthly_patch_size', THEME2_DIR)

  -> Saved: results\theme2_figures\rq4_monthly_patch_size.png


WindowsPath('results/theme2_figures/rq4_monthly_patch_size.png')

## RQ5: How does revision burden change over time?

In [10]:
# Monthly revision rate: % of merged PRs that had >1 commit
agent_rev = rev_stats[rev_stats['agent'].isin(AGENTS)].copy()
agent_rev['is_revised'] = agent_rev['num_commits'] > 1

rows_rev = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = agent_rev[(agent_rev['month'] == m) & (agent_rev['agent'] == agent)]
        row[agent] = round(sub['is_revised'].mean() * 100, 1) if len(sub) >= 5 else None
    rows_rev.append(row)
monthly_rev = pd.DataFrame(rows_rev).set_index('month')
print('Monthly revision rate (%):')
print(monthly_rev.to_string())

Monthly revision rate (%):
         Copilot  Cursor  Claude_Code  Devin
month                                       
2024-12     32.2    44.5         26.5   55.1
2025-01     37.7    42.7         32.7   56.8
2025-02     35.8    43.3         28.7   43.1
2025-03     32.2    46.4         24.6   40.6
2025-04     50.0    49.9         34.7   38.7
2025-05     91.0    41.3         33.9   41.7
2025-06     91.6    36.6         36.7   39.6
2025-07     91.6    35.8         43.7   39.7
2025-08     96.3    43.0         39.4   46.1
2025-09     93.9    46.8         39.9   62.5
2025-10     94.1    46.8         38.6   61.0
2025-11     92.9    46.8         37.3   65.8
2025-12     90.7    47.2         34.3   56.1
2026-01     90.4    47.4         35.1   47.8
2026-02     91.1    49.7         35.0   65.2


In [11]:
# Figure: monthly revision rate
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_rev.index, monthly_rev[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Revision Rate (%)')
ax.set_title('RQ5: Monthly Revision Rate per Agent')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq5_monthly_revision_rate', THEME2_DIR)

  -> Saved: results\theme2_figures\rq5_monthly_revision_rate.png


WindowsPath('results/theme2_figures/rq5_monthly_revision_rate.png')

In [12]:
# Monthly median revision lines added (for revised PRs only)
rows_revsize = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = agent_rev[(agent_rev['month'] == m) & (agent_rev['agent'] == agent)
                        & (agent_rev['num_commits'] > 1)]
        row[agent] = round(sub['rev_lines_added'].median(), 1) if len(sub) >= 5 else None
    rows_revsize.append(row)
monthly_revsize = pd.DataFrame(rows_revsize).set_index('month')

fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_revsize.index, monthly_revsize[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Revision Lines Added')
ax.set_title('RQ5: Monthly Median Revision Effort (Lines Added in Revisions)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq5_monthly_revision_effort', THEME2_DIR)

  -> Saved: results\theme2_figures\rq5_monthly_revision_effort.png


WindowsPath('results/theme2_figures/rq5_monthly_revision_effort.png')